<a href="https://colab.research.google.com/github/aymanr8-dot/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026_09_18%20-%20pandas_challenge-%20lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [2]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [3]:
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()
print(f"Rows: {len(df)}")
print(f"Total revenue: ${total_revenue:,.2f}")
print(f"Total units: {total_units}")
print(f"Across {len(df)} orders the vendors sold {total_units} units generating ${total_revenue:,.2f} in revenue.")

Rows: 400
Total revenue: $8,520.00
Total units: 783
Across 400 orders the vendors sold 783 units generating $8,520.00 in revenue.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [4]:
by_category = df.groupby('category', as_index=False)['revenue'].sum()
by_category = by_category.sort_values('revenue', ascending=False).reset_index(drop=True)
by_category['share_pct'] = (by_category['revenue'] / by_category['revenue'].sum() * 100).round(1)
print(by_category)
top = by_category.iloc[0]
print(f"{top['category']} leads with ${top['revenue']:,.2f} ({top['share_pct']}% of total revenue).")

   category  revenue  share_pct
0      Food   4293.0       50.4
1     Merch   1771.5       20.8
2     Drink   1554.0       18.2
3  RainGear    901.5       10.6
Food leads with $4,293.00 (50.4% of total revenue).


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [5]:
by_vendor = df.groupby('vendor_id')['revenue'].agg(avg_revenue='mean', order_count='count')
by_vendor = by_vendor.sort_values('avg_revenue', ascending=False)
by_vendor['avg_revenue'] = by_vendor['avg_revenue'].round(2)
print(by_vendor)
best = by_vendor.index[0]
print(f"{best} has the highest average order revenue at ${by_vendor.loc[best,'avg_revenue']:,.2f} across {by_vendor.loc[best,'order_count']} orders.")

           avg_revenue  order_count
vendor_id                          
V-01             22.60           94
V-18             21.75          108
V-05             20.58           93
V-10             20.31          105
V-01 has the highest average order revenue at $22.60 across 94 orders.


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [6]:
merch_share = df.loc[df['category'] == 'Merch', 'revenue'].sum() / df['revenue'].sum() * 100
merch_share = round(merch_share, 1)
print(f"{merch_share}%")
print(f"Merch accounts for {merch_share}% of total revenue.")

20.8%
Merch accounts for 20.8% of total revenue.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [7]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')
print(f"Rows before: {len(df)}, after: {len(joined)}")
print(f"Revenue before: {df['revenue'].sum():.2f}, after: {joined['revenue'].sum():.2f}")
unmatched = joined.loc[joined['vendor_name'].isna(), 'vendor_id'].unique()
print(f"Unmatched vendor id(s): {list(unmatched)}")
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown (' + joined['vendor_id'] + ')')

Rows before: 400, after: 400
Revenue before: 8520.00, after: 8520.00
Unmatched vendor id(s): ['V-18']


**The unmatched vendor, and what I did about it: V-18 was the unmatched vednor ID. Instead of dropping it from the data and possibly losing out on 108 orders, I kept the rows through a left join and filled the name with a placeholder, and flagged that the lookup table needs V-18 added to it.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [8]:
pivot = df.pivot_table(index='vendor_id', columns='category', values='revenue', aggfunc='sum', fill_value=0, margins=True, margins_name='Total')
print(pivot.round(2))

category    Drink    Food   Merch  RainGear   Total
vendor_id                                          
V-01        171.0  1338.0   373.5     241.5  2124.0
V-05        298.5   882.0   489.0     244.5  1914.0
V-10        502.5  1054.5   400.5     175.5  2133.0
V-18        582.0  1018.5   508.5     240.0  2349.0
Total      1554.0  4293.0  1771.5     901.5  8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [9]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) I would recommend the vendors to go forward with changing the product mix to include more Food and Merch instead of RainGear. RainGear only accounted for 10.6% of the total revenue, bringing in \$901.50  of pure dollar value whereas Food itself brought 50.4% of the entire revenue, at \$4293. The shelf space and prep
currently going to RainGear would pay off a lot better if directed towards Food or Merch which both generate a significantly larger portion of Revenue. Furthermore, Vendor V-01 has the highest average order revenue at $22.60 amongst all the vendors, as the pivot table shows that V-01 also prioritized Food in its product mix.



---





b) My answer to Q3 would be the least trustworthy in my opinion. This is due to the fact that the claim that V-01 has the highest average order revenue is correct but only by a small spread of roughly \$2 between the other top vendors. As a result, a few different orders could completely change the ranking. Additionally, later outputs also showed that V-01 has the lowest total revenue at \$2124, which could mean that its product mix overindexing on Food specifically may also be a shortfall as a vendor. The conclusion drawn from the output in Q3 mainly rests on a thin margin without understanding the categories and sales based on product mix, which shows us the importance of knowing more about the data before jumping to certain conclusions.